# ENTERPRISE ML TRAINING - MODULE 01
## Introduction to AI & Linear Regression

**Author:** Enterprise Training Team  
**Version:** 1.0  
**Focus:** Predicting Server Resource Utilization


In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('✓ Libraries imported successfully')
print(f'Pandas: {pd.__version__}, NumPy: {np.__version__}')


## Code Explanation
Imports all necessary libraries for data science and machine learning.

## Expected Output
Confirmation message with library versions.


In [ ]:
df = pd.read_csv('server_resource_utilization_linear_regression_25000.csv')
print(f'✓ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nFirst 5 rows:')
print(df.head())


## Code Explanation
Loads CSV dataset using pandas read_csv() function.

## Expected Output
DataFrame with first 5 records displayed.


In [ ]:
print('DATASET INFORMATION')
print(f'Shape: {df.shape}')
print(f'\nData Types:')
print(df.dtypes)
print(f'\nMissing Values: {df.isnull().sum().sum()}')
print(f'\nBasic Statistics:')
print(df.describe())


## Code Explanation
Displays dataset structure, types, and statistics.

## Expected Output
DataFrame info and descriptive statistics.


In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numerical_cols].corr()['resource_utilization_score'].sort_values(ascending=False)
print('CORRELATION WITH TARGET VARIABLE')
print(corr)


## Code Explanation
Calculates correlations between features and target variable.

## Expected Output
List of features sorted by correlation strength.


In [ ]:
df_model = df.copy()
categorical_cols = df_model.select_dtypes(include=['object']).columns.tolist()
df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

X = df_model.drop('resource_utilization_score', axis=1)
y = df_model['resource_utilization_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set: {len(X_train):,} samples')
print(f'Testing set: {len(X_test):,} samples')
print(f'Features: {X.shape[1]}')


## Code Explanation
Prepares data: encodes categorical variables and splits into train/test sets.

## Expected Output
Confirmation of split sizes and feature count.


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('✓ Model training complete!')
print(f'\nIntercept: {model.intercept_:.4f}')
print(f'Coefficients: {len(model.coef_)}')

coefficients_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('\nTop 10 Most Important Features:')
print(coefficients_df.head(10).to_string(index=False))


## Code Explanation
Trains Linear Regression model and displays coefficients.

## Expected Output
Model intercept, coefficient count, and top features.


In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print('MODEL EVALUATION METRICS')
print(f'\n{"Metric":<20} {"Training":<15} {"Testing":<15}')
print('-' * 50)
print(f'{"R² Score":<20} {train_r2:<15.4f} {test_r2:<15.4f}')
print(f'{"MAE":<20} {train_mae:<15.4f} {test_mae:<15.4f}')
print(f'{"RMSE":<20} {train_rmse:<15.4f} {test_rmse:<15.4f}')

print(f'\n✓ R² = {test_r2:.2%} (explains {test_r2:.0%} of variation)')
print(f'✓ Predictions within ±{test_rmse:.2f} points')


## Code Explanation
Makes predictions and calculates evaluation metrics.

## Expected Output
Table comparing training vs testing performance metrics.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(y_test, y_test_pred, alpha=0.5, s=20)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual')
axes[0, 0].set_ylabel('Predicted')
axes[0, 0].set_title(f'Actual vs Predicted (R²={test_r2:.3f})')
axes[0, 0].grid(True, alpha=0.3)

residuals = y_test - y_test_pred
axes[0, 1].hist(residuals, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Residuals')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Residuals Distribution')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].scatter(y_test_pred, residuals, alpha=0.5, s=20)
axes[1, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Residuals')
axes[1, 0].set_title('Residuals vs Predicted')
axes[1, 0].grid(True, alpha=0.3)

errors = np.abs(y_test - y_test_pred)
axes[1, 1].hist(errors, bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Absolute Error')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title(f'Error Distribution (MAE={test_mae:.3f})')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('✓ Visualizations created')


## Code Explanation
Creates 4 diagnostic plots showing model performance.

## Expected Output
Four visualizations: predictions, residuals, and error distributions.


In [ ]:
joblib.dump(model, 'resource_utilization_model.pkl')
joblib.dump(X.columns.tolist(), 'feature_names.pkl')

print('✓ Model saved to: resource_utilization_model.pkl')
print('✓ Feature names saved to: feature_names.pkl')
print('\n✓ Model ready for production deployment!')


## Code Explanation
Saves trained model and feature names using joblib for later use.

## Expected Output
Confirmation of model files saved.


# SUMMARY & KEY TAKEAWAYS

## What We Accomplished

✅ Loaded 25,000 server records  
✅ Explored data and identified correlations  
✅ Prepared data (encoding, splitting)  
✅ Trained Linear Regression model  
✅ Evaluated performance (R², MAE, RMSE)  
✅ Created diagnostic visualizations  
✅ Deployed model for production  

## Key Concepts

**Linear Regression Formula:**  
ŷ = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ

**Evaluation Metrics:**
- R² = 0.8: Model explains 80% of variation
- RMSE = ±4.5: Predictions within 4.5 points
- MAE = 3.2: Average error is 3.2 points

## Next Steps

1. Try polynomial regression for non-linear patterns
2. Use Ridge/Lasso regularization
3. Build separate models per server type
4. Deploy to cloud platform
5. Set up monitoring and retraining

## Congratulations! 🎉

You've completed Module 01 of Enterprise AI & ML Training!

**Ready for:** Module 02 - Advanced Regression Techniques
